In [115]:
import os
import math
import pandas as pd
from tqdm import tqdm
import numpy as np
import plotly.express as px

In [116]:
HIGHER_CATEGORIES = {
    "Demographic and Personal Characteristics": ['AgeVariable', 'EducationVariable', 'EthnicityVariable', 
                                                'GenderVariable', 'PersonalityVariable', 'HormonesVariable', 
                                                'ReligiosityVariable'],
    
    "Social and Group Dynamics": ['AcquaintanceVariable', 'Group_SizeVariable', 'LeadershipVariable', 
                                  'MonitoringVariable', 'MatchingVariable', 'OstracismVariable', 
                                  'Partner_realVariable', 'Partner_typeVariable', 
                                  'Intergroup_Competition_Variable', 'Perception_of_the_partner(s)Variable', 
                                  "Partner(s)'_strategiesVariable", 'SymmetryVariable'],
    
    "Psychological and Cognitive Factors": ['Cognitive_loadVariable', 'EmotionsVariable', 'ExpectationsVariable', 
                                           'Game_ComprehensionVariable', 'Motivational_OrientationVariable', 
                                           'Preferences_for_Conditional_CooperationVariable', 'State_TrustVariable'],
    
    "Game and Experimental Conditions": ['Behavior_in_the_gameVariable', 'ChoicesVariable', 'Experimental_SettingVariable', 
                                        'FeedbackVariable', 'FramingVariable', 'Game_DurationVariable', 
                                        'Game_TypeVariable', 'PeriodVariable', 'SequentialityVariable', 
                                        'Shadow_of_the_FutureVariable', 'Time_PressureVariable'],
    
    "Institutional and Structural Factors": ['AnchorVariable', 'CriticalityVariable', 'Degree_of_conflicting_interestsVariable', 
                                            'IncentivesVariable', 'Institutional_ChoiceVariable', 'Institution__Variable', 
                                            'TaxationVariable'],
    
    "Game Types and Scenarios": ['Public_Goods_GameVariable', 'Resource_Dilemma_GameVariable', 
                                 'Step-level_Public_Goods_GameVariable'],
    
    "Behavioral and Strategic Factors": ['CommunicationVariable', 'IdentificationVariable', 'Normative_BehaviorVariable', 
                                        'Partner_typeVariable', 'ReputationVariable', 'SynchronyVariable'],
    
    "Control and Influence Mechanisms": ['PowerVariable', 'PunishmentVariable', 'RewardVariable', 'PrimingVariable'],
    
    "Environmental and External Factors": ['Physical_ProximityVariable', 'UncertaintyVariable', 
                                          'Shadow_of_the_FutureVariable']
}

HIGHER_CATEGORIES_R = {y: x for x, l in HIGHER_CATEGORIES.items() for y in l}

In [117]:
def type_of_effect(row):
    """ Categorize effect based on its signifiance """
    if math.isnan(row.ESLower) or math.isnan(row.ESUpper):
        if row.ES > -0.2 and row.ES < 0.2:
            return 'noEffect'
        return 'positive' if row.ES >= 0.2 else 'negative'
    if row.ESLower <= 0 <= row.ESUpper:
        return 'noEffect'
    return 'positive'  if float(row.ES) > 0 else 'negative'

def get_output_csv(path, giv):
    df = pd.read_csv(path, index_col=0)
    df["giv"] = giv
    return df 

def get_input_csv(path):
    df = pd.read_csv(path, index_col=0)
    tqdm.pandas()
    df["effect"] = df.progress_apply(type_of_effect, axis=1)
    return df

In [118]:
THS = ["regular", "study_mod", "var_mod"]
DATA_IN = {
    th: get_input_csv(f"../../data/hypotheses/entry/h_{th}_es_d.csv") for th in THS
}

100%|██████████| 846/846 [00:00<00:00, 71003.13it/s]


In [119]:
def format_data_out(fo):
    files = [x for x in os.listdir(fo) if x.endswith(".csv")] 
    data_out = pd.concat([get_output_csv(path=os.path.join(fo, x), giv=x.replace(".csv", "")) for x in files], axis=0)
    data_out["dependent"]="https://data.cooperationdatabank.org/id/dependentvariable/" + data_out["dependent"]
    data_out["category"] = data_out["giv"].apply(lambda x: HIGHER_CATEGORIES_R[x])
    return data_out

DATA_OUT = {
    th: format_data_out(f"./final/h_{th}_es_d/outputs") for th in THS
}
for k, v in DATA_IN.items():
    print(f"{k}\t# h_data: {v.shape[0]}\t# h_output: {DATA_OUT[k].shape[0]}")


regular	# h_data: 4463	# h_output: 243
study_mod	# h_data: 91156	# h_output: 290
var_mod	# h_data: 846	# h_output: 60


In [120]:
curr_df = DATA_IN["regular"]
cols_filter = ["dependent"] + [col for col in curr_df.columns if col.endswith("_label")]

cols_filter

['dependent', 'iv_label', 'cat_t1_label', 'cat_t2_label']

In [121]:
test = curr_df.groupby(cols_filter+["effect"]).agg({'obs': 'count'}).reset_index()
test.head(2)

,dependent,iv_label,cat_t1_label,cat_t2_label,effect,obs
0,https://data.cooperationdatabank.org/id/depend...,Age cohort,middle,old,negative,1
1,https://data.cooperationdatabank.org/id/depend...,Age cohort,middle,old,noEffect,1


In [122]:
test[(test.iv_label=="Shadow of the future")]

,dependent,iv_label,cat_t1_label,cat_t2_label,effect,obs
356,https://data.cooperationdatabank.org/id/depend...,Shadow of the future,True,False,negative,1
357,https://data.cooperationdatabank.org/id/depend...,Shadow of the future,True,False,positive,2
786,https://data.cooperationdatabank.org/id/depend...,Shadow of the future,True,False,negative,2
787,https://data.cooperationdatabank.org/id/depend...,Shadow of the future,True,False,noEffect,2
788,https://data.cooperationdatabank.org/id/depend...,Shadow of the future,True,False,positive,3
789,https://data.cooperationdatabank.org/id/depend...,Shadow of the future,true,false,noEffect,1


In [146]:
def get_evidence_data(cols, data, row):
    for col in cols:
        data = data[data[col].astype(str).str.lower() == str(row[col]).lower()]
        # if isinstance(row[col], str):
        #     data = data[data[col].str.lower() == row[col].lower()]
        # else:
        #     data = data[data[col] == row[col]]
    return data

def get_ratio(cols, data, row):
    evidence = get_evidence_data(cols, data, row)
    grouped = evidence.groupby('effect').agg({'obs': 'count'}).reset_index()
    if row.comparative == "higher":  # #pos/#neg
        col1, col2 = 'positive', 'negative'
    else:  # row.comparative == "lower":  # #neg/#pos
        col2, col1 = 'positive', 'negative'
    row["evidence_plus"] = (list(grouped[grouped.effect==col1].obs.values) + [0])[0]
    row["evidence_null"] = (list(grouped[grouped.effect=="noEffect"].obs.values) + [0])[0]
    row["evidence_minus"] = (list(grouped[grouped.effect==col2].obs.values) + [0])[0]
    row["evidence"] = row["evidence_plus"] + row["evidence_minus"] + row["evidence_null"]
    row["acc"] = row["evidence_plus"]/(row["evidence_plus"]+row["evidence_minus"]+row["evidence_null"]) if (row["evidence_plus"]+row["evidence_minus"]+row["evidence_null"]) else "N/A"
    row["diff"] = row["evidence_plus"] - row["evidence_minus"]
    return row

def add_info(df, th):
    cols_filter = ["dependent"] + [col for col in df.columns if col.endswith("_label")]
    tqdm.pandas()
    df["th"] = th
    df = df.progress_apply(lambda row: get_ratio(cols_filter, DATA_IN[th], row), axis=1)
    return df

DATA_OUT = {k: add_info(v, k) for k, v in DATA_OUT.items()}
DATA_OUT["regular"].sample(5)


100%|██████████| 60/60 [00:00<00:00, 439.38it/s]


,dependent,iv,iv_label,cat_t1,cat_t1_label,cat_t2,cat_t2_label,comparative,giv,category,th,evidence_plus,evidence_null,evidence_minus,evidence,acc,diff
2,https://data.cooperationdatabank.org/id/depend...,gameIncentive,Game incentive,monetary,Monetary,hypothetical,Hypothetical,higher,IncentivesVariable,Institutional and Structural Factors,regular,9,16,5,30,0.3,4
0,https://data.cooperationdatabank.org/id/depend...,synchrony,Synchrony,True,True,False,False,higher,SynchronyVariable,Behavioral and Strategic Factors,regular,4,0,0,4,1.0,4
0,https://data.cooperationdatabank.org/id/depend...,ageCohort,Age cohort,old,old,young,young,higher,AgeVariable,Demographic and Personal Characteristics,regular,5,0,0,5,1.0,5
2,https://data.cooperationdatabank.org/id/depend...,nbTrialsLevel,Number of trials level,low,Low,high,High,lower,Game_DurationVariable,Game and Experimental Conditions,regular,0,0,1,1,0.0,-1
2,https://data.cooperationdatabank.org/id/depend...,institutionalChoice,Institutional choice,endogenous,Endogenous,exogenous,Exogenous,higher,Institutional_ChoiceVariable,Institutional and Structural Factors,regular,19,0,7,26,0.730769,12


In [147]:
COLS_KEEP = ["giv", "category", "th", "evidence_plus", "evidence_null", "evidence_minus", "evidence", "acc", "diff"]
DATA_VIS = pd.concat([v[COLS_KEEP] for _, v in DATA_OUT.items()], axis=0)
DATA_VIS.sample(3)

,giv,category,th,evidence_plus,evidence_null,evidence_minus,evidence,acc,diff
2,PeriodVariable,Game and Experimental Conditions,study_mod,0,0,1,1,0.0,-1
0,FeedbackVariable,Game and Experimental Conditions,regular,1,2,0,3,0.333333,1
3,EmotionsVariable,Psychological and Cognitive Factors,regular,0,1,0,1,0.0,0


In [148]:
DATA_VIS[DATA_VIS.acc=="N/A"].shape

(23, 9)

In [149]:
DO_MEAN_VAR = DATA_VIS.groupby(["th", "giv", "category"]).agg({"diff": ["mean", "std", ("cv", lambda x: np.std(x)/abs(np.mean(x)) if np.mean(x) else "N/A")]}).reset_index()
DO_MEAN_VAR.columns = ['_'.join(x) for x in list(DO_MEAN_VAR.columns)]

color_palette = px.colors.qualitative.Safe
fig = px.scatter(DO_MEAN_VAR.dropna(), x="diff_mean", y="diff_cv", color="th_", hover_data="giv_",
                 color_discrete_sequence=color_palette)
fig.write_image("../visualisations/llm_mean_diff_vs_coeff_var_diff.pdf", format='pdf')
fig.show()

In [192]:
CURR_DF = DATA_VIS[DATA_VIS.acc != "N/A"]
CURR_DF["perc_null"] = CURR_DF["evidence_null"] / CURR_DF["evidence"]
CURR_DF["perc_plus"] = CURR_DF["evidence_plus"] / CURR_DF["evidence"]
CURR_DF["perc_minus"] = CURR_DF["evidence_minus"] / CURR_DF["evidence"]
CURR_DF.groupby("th").agg({f"perc_{x}": "mean" for x in ["plus", "null", "minus"]})

/var/folders/p9/2gjyjx2x3pjb2m5w91srfdq40000gp/T/ipykernel_20188/2672543262.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/p9/2gjyjx2x3pjb2m5w91srfdq40000gp/T/ipykernel_20188/2672543262.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/p9/2gjyjx2x3pjb2m5w91srfdq40000gp/T/ipykernel_20188/2672543262.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

,perc_plus,perc_null,perc_minus
th,,,
regular,0.278541,0.457696,0.263763
study_mod,0.316405,0.397010,0.286585
var_mod,0.291867,0.501073,0.207061


In [204]:
color_palette = px.colors.qualitative.Safe
fig = px.histogram(DATA_VIS, x="acc", histnorm="", nbins=30, color="th",
                   color_discrete_sequence=color_palette, opacity=0.75)
fig.update_layout(barmode='group', yaxis_title="Count", xaxis_title="Acc")
fig.write_image("../visualisations/llm_hist_acc.pdf", format='pdf')
fig.show()

In [163]:
print(f"Accuracy 1: {DATA_VIS[DATA_VIS.acc==1].shape[0]} | {DATA_VIS[DATA_VIS.acc==1].evidence.unique()}")
print(f"Accuracy 0: {DATA_VIS[DATA_VIS.acc==0].shape[0]} | {DATA_VIS[DATA_VIS.acc==0].evidence.unique()}")

Accuracy 1: 79 | [ 1  3  5  4  7  8  2 10]
Accuracy 0: 271 | [ 1  2  4  3  8  6  7  9  5 17 13 19 11 10]


In [205]:
color_palette = px.colors.qualitative.Safe
fig = px.scatter(DATA_VIS[DATA_VIS.evidence < 100], x="acc", y="evidence", color="th", symbol="th", #size="evidence", size_max=50,
      color_discrete_sequence=color_palette, opacity=0.75)
fig.update_layout(yaxis_title="Evidence Count", xaxis_title="Acc")
fig.write_image("../visualisations/llm_scatter_acc_evidence.pdf", format='pdf')
fig.show()

In [165]:
DATA_VIS.head(2)

,giv,category,th,evidence_plus,evidence_null,evidence_minus,evidence,acc,diff
0,Resource_Dilemma_GameVariable,Game Types and Scenarios,regular,0,1,0,1,0.0,0
1,Resource_Dilemma_GameVariable,Game Types and Scenarios,regular,0,0,2,2,0.0,-2


In [166]:
top = 5
pos_dif = DATA_VIS.groupby("giv").agg({"diff": "mean"}).reset_index().sort_values(by="diff", ascending=False).giv.values[:5]
neg_dif = DATA_VIS.groupby("giv").agg({"diff": "mean"}).reset_index().sort_values(by="diff", ascending=True).giv.values[:5]

In [167]:
color_palette = px.colors.qualitative.Safe
fig = px.box(DATA_VIS[DATA_VIS.giv.isin(pos_dif)], 
             x="giv", y="diff", color="th", points='all',
             color_discrete_sequence=color_palette)
fig.update_layout(yaxis={'categoryorder': 'total descending'})
fig.update_layout(width=600, height=600, showlegend=False)
fig.write_image("../visualisations/llm_pos_diff_giv_all.pdf", format='pdf')
fig.show()

In [168]:
color_palette = px.colors.qualitative.Safe
fig = px.box(DATA_VIS[DATA_VIS.giv.isin(neg_dif)], 
             x="giv", y="diff", color="th", points='all',
             color_discrete_sequence=color_palette)
fig.update_layout(yaxis={'categoryorder': 'total descending'})
fig.update_layout(width=600, height=600, showlegend=False)
fig.write_image("../visualisations/llm_neg_diff_giv_all.pdf", format='pdf')

fig.show()

In [169]:
top = 5
pos_dif_all = []
neg_diff_all = []
data_vis_pos = []
data_vis_neg = []
for th in DATA_VIS.th.unique():
    pos_diff = DATA_VIS[DATA_VIS.th==th].groupby("giv").agg({"diff": "mean"}).reset_index().sort_values(by="diff", ascending=False).giv.values.tolist()[:5]
    neg_diff = DATA_VIS[DATA_VIS.th==th].groupby("giv").agg({"diff": "mean"}).reset_index().sort_values(by="diff", ascending=True).giv.values.tolist()[:5]
    data_vis_pos.append(DATA_VIS[(DATA_VIS.th==th) & (DATA_VIS.giv.isin(pos_diff))])
    data_vis_neg.append(DATA_VIS[(DATA_VIS.th==th) & (DATA_VIS.giv.isin(neg_diff))])

In [170]:
color_palette = px.colors.qualitative.Safe
fig = px.box(pd.concat(data_vis_pos, axis=0), 
             x="giv", y="diff", color="th", points='all',
             color_discrete_sequence=color_palette)
fig.update_layout(yaxis={'categoryorder': 'total descending'})
fig.update_layout(width=600, height=600, showlegend=False)
fig.write_image("../visualisations/llm_pos_diff_giv_distinct_per_th.pdf", format='pdf')
fig.show()

In [171]:
color_palette = px.colors.qualitative.Safe
fig = px.box(pd.concat(data_vis_neg, axis=0), 
             x="giv", y="diff", color="th", points='all',
             color_discrete_sequence=color_palette)
fig.update_layout(yaxis={'categoryorder': 'total descending'})
fig.update_layout(width=600, height=600, showlegend=False)
fig.write_image("../visualisations/llm_neg_diff_giv_distinct_per_th.pdf", format='pdf')
fig.show()